# Aarogya — Extensive Fine-tuning of Gemma 4 / 3 (Unsloth + QLoRA)

**Hackathon:** Gemma 4 Good · Special Track: **Unsloth** ($10K)

**Hardware:** Kaggle GPU **T4 × 2** · Internet **ON**

**Runtime:** ~2-3 hours for 1500 steps on 5000 samples

Auto-falls-back to Gemma 3 4B if Unsloth hasn't published a Gemma 4 4B build yet.

## 1. Install Unsloth (uses prebuilt wheels — no source compilation)

In [ ]:
%%capture
# Unsloth's official install pattern — let pip pick compatible bitsandbytes/triton/xformers
!pip install -q --upgrade pip
!pip install -q unsloth
# Force latest Unsloth (often has critical Gemma fixes that aren't on PyPI yet)
!pip uninstall -q -y unsloth unsloth_zoo
!pip install -q --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git git+https://github.com/unslothai/unsloth-zoo.git

In [ ]:
# Sanity check — fail fast if Unsloth didn't install
from unsloth import FastLanguageModel
import torch
print('Unsloth OK · GPUs:', torch.cuda.device_count(), '· CUDA:', torch.cuda.is_available())

## 2. Clone the Aarogya repo + build dataset

In [ ]:
import os
if not os.path.exists('/kaggle/working/aarogya'):
    !git clone https://github.com/shabdkumar3/aarogya.git /kaggle/working/aarogya
%cd /kaggle/working/aarogya/finetuning
!python prepare_dataset.py
!ls -la train_data.jsonl val_data.jsonl

## 3. Pick the best Gemma model available on Unsloth (Gemma 4 → Gemma 3 fallback)

In [ ]:
from huggingface_hub import HfApi
from huggingface_hub.utils import RepositoryNotFoundError

CANDIDATES = [
    'unsloth/gemma-4-4b-it',
    'unsloth/gemma-4-4b-it-bnb-4bit',
    'unsloth/gemma-3-4b-it',
    'unsloth/gemma-3-4b-it-bnb-4bit',
]
api = HfApi()
MODEL_NAME = None
for m in CANDIDATES:
    try:
        api.model_info(m)
        MODEL_NAME = m
        break
    except RepositoryNotFoundError:
        continue
if MODEL_NAME is None:
    raise RuntimeError('No Gemma 4/3 4B model found on Unsloth. Check https://huggingface.co/unsloth')
print('Using:', MODEL_NAME)

## 4. Load model with QLoRA (4-bit)

In [ ]:
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

## 5. Train (1500 steps · ~2-3h on T4×2)

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

train_ds = load_dataset('json', data_files='train_data.jsonl', split='train')
val_ds   = load_dataset('json', data_files='val_data.jsonl',   split='train')
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=50,
        max_steps=1500,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        eval_strategy='steps',
        eval_steps=150,
        save_strategy='steps',
        save_steps=500,
        save_total_limit=2,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='/kaggle/working/aarogya-checkpoints',
        report_to='none',
    ),
)
stats = trainer.train()
print(stats)

## 6. Save LoRA adapter

In [ ]:
model.save_pretrained('/kaggle/working/aarogya-lora')
tokenizer.save_pretrained('/kaggle/working/aarogya-lora')
!ls -lh /kaggle/working/aarogya-lora

## 7. Export merged GGUF for Ollama (optional but nice for Ollama track)

In [ ]:
try:
    model.save_pretrained_gguf(
        '/kaggle/working/aarogya-gguf',
        tokenizer,
        quantization_method='q4_k_m',
    )
    !ls -lh /kaggle/working/aarogya-gguf
except Exception as e:
    print('GGUF export skipped:', e)

## 8. Sample inference — fine-tuned vs base (for the writeup)

In [ ]:
FastLanguageModel.for_inference(model)
test_prompts = [
    'Bachcha 2 din se paani-wala dast, kuch khaya nahi, sust hai',
    'Patient has fever 102F for 3 days with body pain and headache',
    'Aurat 3 mahine pregnant, blood pressure high, sir mein dard',
]
for p in test_prompts:
    inputs = tokenizer([f'<start_of_turn>user\nSymptoms: {p}<end_of_turn>\n<start_of_turn>model\n'], return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=200, use_cache=True)
    print('PROMPT:', p)
    print('OUTPUT:', tokenizer.decode(out[0], skip_special_tokens=True))
    print('-' * 60)